# Bach Violin Dataset — Timbre Feature Distribution

Extract per-frame timbre features from every audio file in the Bach violin dataset
using `TimbreMetrics.extract_series_from_file`, concatenate all frames into a single
feature matrix, and save it for downstream use (e.g. as a reference distribution for evaluation).

**Output**: `data/processed/bach_violin_timbre_features.parquet` and `.npz`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation.timbre_metrics import TimbreMetrics

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
AUDIO_DIR  = PROJECT_ROOT / "data" / "raw" / "bach-violin" / "audio"
AUDIO_CSV  = PROJECT_ROOT / "data" / "raw" / "bach-violin" / "audio.csv"
OUT_DIR    = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# TimbreMetrics config
SR             = 16_000
FRAME_WIDTH    = 0.25   # seconds
OVERLAP        = 0.0
FEATURE_TYPES  = ["spectral"]   # or ["spectral", "level", "harmonic"]

print(f"Audio dir: {AUDIO_DIR}  (exists: {AUDIO_DIR.exists()})")
print(f"Audio CSV: {AUDIO_CSV}  (exists: {AUDIO_CSV.exists()})")

## 1. Discover audio files and load metadata

In [ ]:
audio_files = sorted(
    [p for p in AUDIO_DIR.rglob("*") if p.suffix.lower() in {".mp3", ".opus", ".wav"}]
)
print(f"Found {len(audio_files)} audio files")

# Load CSV for metadata lookup keyed by filename stem
meta_df = pd.read_csv(AUDIO_CSV)
# Build lookup: filename (no parent path) -> row
meta_lookup = meta_df.set_index("filename").to_dict(orient="index")
print(f"Metadata rows: {len(meta_df)}")
meta_df.head()

## 2. Extract per-frame features for every file

In [ ]:
tm = TimbreMetrics(sample_rate=SR, frame_width_sec=FRAME_WIDTH, overlap_pct=OVERLAP)

all_features: list[dict] = []   # one dict per file: {feat: array, + metadata}
failed: list[str] = []

for path in tqdm(audio_files, desc="Extracting"):
    try:
        series = tm.extract_series_from_file(path, feature_types=FEATURE_TYPES)
    except Exception as e:
        print(f"  SKIP {path.name}: {e}")
        failed.append(path.name)
        continue

    if not series:
        failed.append(path.name)
        continue

    meta = meta_lookup.get(path.name, {})
    all_features.append({
        "filename":   path.name,
        "collection": meta.get("collection", "unknown"),
        "violinist":  meta.get("violinist",  "unknown"),
        "work":       meta.get("work",       "unknown"),
        "series":     series,
        "n_frames":   len(next(iter(series.values()))),
    })

print(f"\nSuccessful: {len(all_features)} / {len(audio_files)}  |  Failed: {len(failed)}")
if failed:
    print("Failed files:", failed)

## 3. Concatenate into a flat DataFrame

In [ ]:
# Keep only feature keys present in every file
common_keys = sorted(
    set.intersection(*[set(r["series"].keys()) for r in all_features])
)
print(f"Features ({len(common_keys)}): {common_keys}")

rows = []
for r in all_features:
    n = r["n_frames"]
    meta_block = pd.DataFrame({
        "filename":   [r["filename"]]   * n,
        "collection": [r["collection"]] * n,
        "violinist":  [r["violinist"]]  * n,
        "work":       [r["work"]]       * n,
    })
    feat_block = pd.DataFrame(
        {k: r["series"][k] for k in common_keys}
    )
    rows.append(pd.concat([meta_block, feat_block], axis=1))

df = pd.concat(rows, ignore_index=True)
print(f"\nDataset shape: {df.shape}  ({df.shape[0]:,} frames × {df.shape[1]} columns)")
df.head()

## 4. Save outputs

In [ ]:
parquet_path = OUT_DIR / "bach_violin_timbre_features.parquet"
npz_path     = OUT_DIR / "bach_violin_timbre_features.npz"

# Parquet — keeps metadata columns alongside feature columns
df.to_parquet(parquet_path, index=False)
print(f"Saved parquet: {parquet_path}  ({parquet_path.stat().st_size / 1e6:.1f} MB)")

# NPZ — feature matrix + feature names + per-frame string metadata
feat_matrix = df[common_keys].to_numpy(dtype=np.float64)
np.savez_compressed(
    npz_path,
    features     = feat_matrix,
    feature_names= np.array(common_keys),
    filename     = df["filename"].to_numpy(),
    collection   = df["collection"].to_numpy(),
    violinist    = df["violinist"].to_numpy(),
    work         = df["work"].to_numpy(),
)
print(f"Saved npz:     {npz_path}  ({npz_path.stat().st_size / 1e6:.1f} MB)")
print(f"Feature matrix shape: {feat_matrix.shape}")

## 5. Quick sanity check — feature distributions per violinist

In [ ]:
import matplotlib.pyplot as plt

print("Frames per violinist:")
display(df.groupby("violinist").size().rename("n_frames").to_frame())

print("\nFeature summary (all frames):")
display(df[common_keys].describe().T.round(4))

# Distribution plot for spectral_centroid across violinists
plot_feat = common_keys[0]  # first feature
fig, ax = plt.subplots(figsize=(12, 4))
for viol, grp in df.groupby("violinist"):
    grp[plot_feat].plot.kde(ax=ax, label=viol)
ax.set_xlabel(plot_feat)
ax.set_title(f"Distribution of '{plot_feat}' per violinist")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()